# Risco e retorno da empresa do seu grupo
**ED0139 · Finanças Corporativas · 2026-2 · Prof. Sérgio Cardoso · UFC**

Versão do aluno. Entrega pelo SIGAA até 02/09.

Este notebook refaz, em Python, exatamente a mesma conta da planilha. A regra da disciplina
é que os dois caminhos cheguem ao mesmo número: método diferente com resultado igual é a
defesa contra erro de fórmula e contra copiar sem entender.

O que você vai calcular:

1. o retorno diário da sua empresa a partir dos preços de fechamento;
2. o retorno médio e o desvio-padrão, ao dia e ao ano;
3. a covariância e a correlação com o BOVA11, o índice inteiro;
4. o risco de uma carteira que junta as duas coisas.

Os dados são do pacote congelado da disciplina: fechamentos oficiais da B3 entre
20/08/2021 e 20/08/2026, sem ajuste por proventos.

## Passo 0 · Preparação

Deixe o arquivo de preços na mesma pasta do notebook, ou ajuste o caminho abaixo.
E escreva o ticker do seu grupo: é a única coisa que muda de um grupo para outro.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CAMINHO = "dados/precos/fechamentos_diarios_2021-2026_congelado_2026-08-24.csv"
TICKER  = "ABEV3"   # <<< o ticker do SEU grupo
PREGOES_ANO = 252

precos = pd.read_csv(CAMINHO, parse_dates=["data"]).set_index("data")
precos[[TICKER, "BOVA11"]].tail()

## Passo 1 · O cuidado que vem antes da conta

Antes de qualquer estatística, olhe os maiores movimentos da série e pergunte de cada um se
foi mercado ou se foi evento societário. Grupamento e desdobramento mudam a unidade do preço
sem mudar a riqueza de ninguém, e um único dia mal tratado contamina cinco anos de conta.

In [ ]:
variacao = precos[TICKER].pct_change()
variacao.abs().sort_values(ascending=False).head(5).map(lambda x: f"{x:.1%}")

**A Hapvida fez grupamento de 15 para 1 em 06/06/2025.** O grupo da HAPV3 precisa corrigir
os preços anteriores a essa data, multiplicando por 15. Os demais grupos deixam `FATOR = 1`,
porque nenhuma das outras sete empresas teve evento do tipo no período.

In [ ]:
DATA_EVENTO = "2025-06-06"
FATOR = 1           # HAPV3: 15. Demais empresas: 1.

serie = precos[TICKER].copy()
if FATOR != 1:
    serie.loc[:DATA_EVENTO] = ...          # corrija os preços anteriores ao evento
    serie.loc[DATA_EVENTO] = ...           # o próprio dia ex já vem agrupado

serie.tail()

## Passo 2 · Retorno

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

O primeiro dia da série não tem retorno, porque não existe preço anterior. É por isso que
1.248 preços produzem 1.247 retornos.

In [ ]:
retorno = ...              # dica: pct_change() e depois dropna()
retorno_ibov = ...

print(f"{len(serie)} preços produziram {len(retorno)} retornos")
retorno.head(3)

## Passo 3 · Retorno médio e risco

$$\bar{R} = \frac{1}{n}\sum_{t=1}^{n} R_t \qquad
s^2 = \frac{1}{n-1}\sum_{t=1}^{n}(R_t-\bar{R})^2 \qquad s = \sqrt{s^2}$$

O denominador $n-1$ existe porque a média foi estimada da própria amostra. Em `pandas`,
`.std()` já usa $n-1$ por padrão; em `numpy`, é preciso pedir `ddof=1`.

Anualização: a média multiplica por 252 e o desvio multiplica pela **raiz** de 252.

In [ ]:
media_dia = ...
desvio_dia = ...

media_ano = ...
desvio_ano = ...

print(f"{TICKER}")
print(f"  retorno médio ao dia .... {media_dia:.4%}")
print(f"  desvio ao dia ........... {desvio_dia:.3%}")
print(f"  retorno médio ao ano .... {media_ano:.2%}")
print(f"  desvio ao ano ........... {desvio_ano:.2%}")

### Confira contra a planilha

Escreva abaixo os dois números que a **sua planilha** produziu. Se a diferença passar de
0,01 ponto percentual, um dos dois caminhos está errado, e achar qual é parte da tarefa.

In [ ]:
EXCEL_RETORNO_ANO = ...    # copie da sua planilha
EXCEL_DESVIO_ANO  = ...

print(f"retorno: Python {media_ano:.2%} × Excel {EXCEL_RETORNO_ANO:.2%}")
print(f"desvio:  Python {desvio_ano:.2%} × Excel {EXCEL_DESVIO_ANO:.2%}")

## Passo 4 · A carteira com o índice

$$\sigma_{12} = \frac{1}{n-1}\sum (R_{1t}-\bar{R_1})(R_{2t}-\bar{R_2})
\qquad \rho = \frac{\sigma_{12}}{\sigma_1\sigma_2}$$

$$\sigma_p^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2w_1w_2\sigma_{12}$$

Os dois primeiros termos nunca são negativos. Só o terceiro pode ser, e é ele que permite
ao risco da carteira cair abaixo do risco dos dois ativos que a compõem.

In [ ]:
par = pd.concat([retorno, retorno_ibov], axis=1).dropna()
par.columns = [TICKER, "BOVA11"]

cov_ano = ...              # dica: par.cov(ddof=1)
corr = ...
desvio_ibov_ano = ...

print(f"covariância ao ano ... {cov_ano:.6f}")
print(f"correlação ........... {corr:.3f}")
print(f"desvio do BOVA11 ..... {desvio_ibov_ano:.2%}")

In [ ]:
def risco_carteira(w, s1, s2, cov):
    """Desvio-padrão de uma carteira de dois ativos, com peso w no primeiro."""
    termo1 = ...
    termo2 = ...
    termo3 = ...
    return np.sqrt(termo1 + termo2 + termo3), (termo1, termo2, termo3)

w = 0.5
dp_carteira, termos = risco_carteira(w, desvio_ano, desvio_ibov_ano, cov_ano)

print(f"termo 1 (a sua empresa) .. {termos[0]:.6f}")
print(f"termo 2 (o índice) ....... {termos[1]:.6f}")
print(f"termo 3 (o cruzado) ...... {termos[2]:.6f}")
print(f"desvio da carteira ....... {dp_carteira:.2%}")

## Passo 5 · O desenho

O gráfico abaixo varre todos os pesos possíveis entre 0% e 100% na sua empresa e mostra o
risco de cada carteira. Se a curva tiver barriga, existe uma combinação menos arriscada do
que ficar só no índice.

In [ ]:
pesos = np.linspace(0, 1, 201)
riscos = np.array([...])       # o risco de cada carteira
w_min = ...                    # o peso que minimiza o risco

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(pesos * 100, riscos * 100, color="#005386", linewidth=2.2)
ax.set_xlabel(f"peso em {TICKER} (%)")
ax.set_ylabel("desvio-padrão da carteira ao ano (%)")
ax.set_title("Risco da carteira conforme o peso", loc="left", fontsize=12)
ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## Passo 6 · A leitura

Responda em texto, no máximo cinco linhas cada, na célula abaixo.

1. O desvio da sua empresa é maior ou menor que o do índice inteiro? Por que isso acontece?
2. A carteira meio a meio ficou menos arriscada que a sua empresa sozinha? O que na fórmula
   explica o resultado?
3. Se a correlação com o índice fosse mais alta, o ganho de juntar as duas seria maior ou
   menor?

*(escreva aqui)*

1.

2.

3.

## O que entregar

Este notebook executado, com todas as saídas visíveis, e a planilha preenchida. Os números
dos dois têm de bater. Erro honesto e documentado vale mais que resultado certo sem rastro.

*Dados: pacote congelado da disciplina. Fontes primárias: B3 e CVM.*